# 每日股價爬蟲！

今天我們要來建立一個股票爬蟲，最後的成品如下：


In [ ]:
import requests
import pandas as pd
from io import StringIO


def crawl_price(date):

    # 將 date 變成字串 舉例：'20260605'
    datestr = date.strftime('%Y%m%d')

    # 從網站上依照 datestr 將指定日期的股價抓下來
    r = requests.post(
        'http://www.twse.com.tw/exchangeReport/MI_INDEX?response=csv&date='
        + datestr
        + '&type=ALLBUT0999'
    )

    # 將抓下來的資料（r.text），其中的等號給刪除
    content = r.text.replace('=', '')

    # 將 column 數量小於等於 10 的行數都刪除
    lines = content.split('\n')
    lines = list(filter(lambda l: len(l.split('",')) > 10, lines))

    # 將每一行再合成同一行，並用肉眼看不到的換行符號'\n'分開
    content = "\n".join(lines)

    # 假如沒下載到，則回傳None（代表抓不到資料）
    if content == '':
        return None

    # 將content變成檔案：StringIO，並且用pd.read_csv將表格讀取進來
    df = pd.read_csv(StringIO(content))

    # 將表格中的元素都換成字串，並把其中的逗號刪除
    df = df.astype(str)
    df = df.apply(lambda s: s.str.replace(',', ''))

    # 將爬取的日期存入 dataframe
    df['date'] = pd.to_datetime(date)

    # 將「證券代號」的欄位改名成「stock_id」
    df = df.rename(columns={'證券代號': 'stock_id'})

    # 將 「stock_id」與「date」設定成index
    df = df.set_index(['stock_id', 'date'])

    # 將所有的表格元素都轉換成數字，error='coerce'的意思是說，假如無法轉成數字，則用 NaN 取代
    df = df.apply(lambda s: pd.to_numeric(s, errors='coerce'))

    # 刪除不必要的欄位
    df = df[df.columns[df.isnull().all() == False]]

    return df


import datetime

crawl_price(datetime.datetime(2026, 6, 5))

,,成交股數,成交筆數,成交金額,開盤價,最高價,最低價,收盤價,漲跌價差,最後揭示買價,最後揭示買量,最後揭示賣價,最後揭示賣量,本益比
stock_id,date,,,,,,,,,,,,,
00400A,2026-06-05,60780296,14112,877124134,14.64,14.74,14.08,14.55,0.31,14.54,190,14.55,394,0.00
00401A,2026-06-05,17322548,1412,235236888,13.80,13.80,13.38,13.68,0.16,13.67,154,13.68,225,0.00
00403A,2026-06-05,545746967,128975,5646466276,10.44,10.54,10.13,10.41,0.18,10.41,1031,10.42,1511,0.00
0050,2026-06-05,187031371,355382,19494259576,105.00,105.35,102.80,104.15,1.95,104.15,1046,104.20,272,0.00
0051,2026-06-05,120899,761,17204042,144.85,144.85,139.45,143.25,2.35,142.85,1,143.25,1,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9944,2026-06-05,52248,49,885172,16.90,17.10,16.90,16.95,0.05,16.90,14,17.05,3,339.00
9945,2026-06-05,10439356,4023,261517700,25.30,25.45,24.60,25.15,0.05,25.15,162,25.20,33,7.06
9946,2026-06-05,1646474,563,28222065,17.10,17.25,16.95,17.15,0.20,17.15,29,17.20,34,4.12


# 接下來就來一步步分析

首先呢，必須知道網址在哪裡，把網址上的資料存成csv檔


In [1]:
import requests

response = requests.get(
    'https://www.twse.com.tw/exchangeReport/MI_INDEX?response=csv&date=20260605&type=ALLBUT0999&_=1520785530355'
)

# 試試看csv能不能直接存到 pandas 的 dataframe 中

發現有點小問題，由於pandas發現每row的長度不一樣，造成pandas無法存取


In [2]:
lines = response.text.split('\n')
lines[100]

'"特選臺灣IC設計動能指數","13,174.51","-","557.07","-4.06","",\r'

# 用For 迴圈篩選每一行


In [5]:
# 將 newlines 檢查是否有 17個欄位，例如：
'"你",   "好",   "嗎",'

# 我們想要把它被切開並計算切開後字串被分割成幾個元素，用眼睛看，我們知道上述這行有3個欄位
# 我們不能直接用「,」來切開字串，因為我們假如考慮以下狀況：
'  "你",   ",好,",   "嗎",'
# 上述狀況，正確來說，我們應該要切開三個值（以被「"」包起來和「,」作為判斷），分別為「你」、「,好,」、「嗎」
# 假如直接用「,」分開的話，反而會切成「你」「"」「好」「,」「嗎」，總共五段
# 所以我們發現以「",」來切開字串，就能準確的將上述字串切開，所以在切割時，我們用「",」將每一行切開，並看切成幾個
# 切成17個的話，我們就保留

newlines = []

for line in lines:

    # 用「",」切開每一行，看是否被切成17個
    if len(line.split('",')) == 17:

        # 將 line 加到新的 newlines 中
        newlines.append(line)

print('原本的行數（lines）')
print(len(lines))
print('刪除不需要的行數後，變少了(newlines)')
print(len(newlines))
print(newlines[1])

原本的行數（lines）
1682
刪除不需要的行數後，變少了(newlines)
1363
="00400A","主動國泰動能高息","60,780,296","14,112","877,124,134","14.64","14.74","14.08","14.55","-","0.31","14.54","190","14.55","394","0.00",


# 終於做出dataframe 了！


In [29]:
# 先創造一個字元c(換行符)
c = '\n'
# 利用此字元c，將每一行給連在一起
s = c.join(newlines)
# 將 s 裡面的 等號 刪除
s = s.replace('=', '')

# 將 s 用StringIO變成檔案，並用 pd.read_csv 來讀取檔案
df = pd.read_csv(StringIO(s))

# 顯示前五個
df.head()

,證券代號,證券名稱,成交股數,成交筆數,成交金額,開盤價,最高價,最低價,收盤價,漲跌(+/-),漲跌價差,最後揭示買價,最後揭示買量,最後揭示賣價,最後揭示賣量,本益比,Unnamed: 16
0,00400A,主動國泰動能高息,"60,780,296","14,112","877,124,134",14.64,14.74,14.08,14.55,-,0.31,14.54,190,14.55,394,0.00,NaN
1,00401A,主動摩根台灣鑫收,"17,322,548","1,412","235,236,888",13.80,13.80,13.38,13.68,-,0.16,13.67,154,13.68,225,0.00,NaN
2,00403A,主動統一升級50,"545,746,967","128,975","5,646,466,276",10.44,10.54,10.13,10.41,-,0.18,10.41,"1,031",10.42,"1,511",0.00,NaN
3,0050,元大台灣50,"187,031,371","355,382","19,494,259,576",105.00,105.35,102.80,104.15,-,1.95,104.15,"1,046",104.20,272,0.00,NaN
4,0051,元大中型100,"120,899",761,"17,204,042",144.85,144.85,139.45,143.25,-,2.35,142.85,1,143.25,1,0.00,NaN


# 用 pandas 中的好用 function，將資料作整理！

上面的資料有點怪怪的，例如：

1. 它們顯示起來像是數字，但其實還是字串！
2. 某些數字中間有','，很煩！
3. 有幾行是來亂的：Unnamed: 16，啥玩意兒？


In [30]:
# 將所有df中的元素都變成字串，並將字串中的逗號「,」刪除
df = df.astype(str)
df = df.apply(lambda s: s.str.replace(',', ''))

# 將 df 證券代號變成 index
df = df.set_index('證券代號')

# 將 df 中的元素從字串變成數字
df = df.apply(lambda s: pd.to_numeric(s, errors='coerce'))

# 要刪除沒有用的columns
# 其中 axis=1 為是說每條columns去檢查有沒有NaN
# how='all' 是說假如全部都是 NaN 則刪除該 column
# （原本的方法） df = df[df.columns[df.isnull().sum() != len(df)]]

df.dropna(axis=1, how='all', inplace=True)

df.head()  # 預設顯示5筆

,成交股數,成交筆數,成交金額,開盤價,最高價,最低價,收盤價,漲跌價差,最後揭示買價,最後揭示買量,最後揭示賣價,最後揭示賣量,本益比
證券代號,,,,,,,,,,,,,
00400A,60780296,14112,877124134,14.64,14.74,14.08,14.55,0.31,14.54,190,14.55,394,0.0
00401A,17322548,1412,235236888,13.80,13.80,13.38,13.68,0.16,13.67,154,13.68,225,0.0
00403A,545746967,128975,5646466276,10.44,10.54,10.13,10.41,0.18,10.41,1031,10.42,1511,0.0
0050,187031371,355382,19494259576,105.00,105.35,102.80,104.15,1.95,104.15,1046,104.20,272,0.0
0051,120899,761,17204042,144.85,144.85,139.45,143.25,2.35,142.85,1,143.25,1,0.0


In [24]:
df.loc['0050']

成交股數      1.870314e+08
成交筆數      3.553820e+05
成交金額      1.949426e+10
開盤價       1.050000e+02
最高價       1.053500e+02
最低價       1.028000e+02
收盤價       1.041500e+02
漲跌價差      1.950000e+00
最後揭示買價    1.041500e+02
最後揭示買量    1.046000e+03
最後揭示賣價    1.042000e+02
最後揭示賣量    2.720000e+02
本益比       0.000000e+00
Name: 0050, dtype: float64

# 計算長紅棒


In [32]:
# 紅棒的長度，1代表不漲不跌，小於一代表收盤價比較小（股價跌），大於一代表收盤價比較大（股票漲）
close_open = df['收盤價'] / df['開盤價']
close_open.head(5)

證券代號
00400A    0.993852
00401A    0.991304
00403A    0.997126
0050      0.991905
0051      0.988954
dtype: float64

In [33]:
# 選出 收盤 比 開盤 還要高 5% 以上的股票
df[close_open > 1.05]

,成交股數,成交筆數,成交金額,開盤價,最高價,最低價,收盤價,漲跌價差,最後揭示買價,最後揭示買量,最後揭示賣價,最後揭示賣量,本益比
證券代號,,,,,,,,,,,,,
1414,345630,345,5688466,16.00,16.90,15.85,16.90,0.80,16.80,4,16.95,7,58.28
1568,548746,439,23320065,40.60,44.70,40.60,43.80,1.25,43.60,1,43.80,3,32.69
2025,2025762,1276,26464016,12.35,13.40,12.30,13.40,1.20,13.40,1323,NaN,0,83.75
2059,1330055,7286,7179872775,5250.00,5805.00,4945.00,5620.00,270.00,5620.00,1,5640.00,2,49.54
2241,3727155,1743,128343040,32.50,35.40,31.85,35.40,3.20,35.40,493,NaN,0,0.00
2327,29037062,60281,21174694565,708.00,778.00,670.00,769.00,26.00,769.00,22,770.00,149,60.55
2375,3597799,4735,528839840,142.50,156.00,132.00,156.00,11.00,155.50,20,156.00,41,47.42
2428,3351311,4306,921504514,263.00,287.00,257.00,283.00,18.00,283.00,11,283.50,12,25.73
2461,9408753,2763,182296054,18.00,19.50,18.00,19.50,1.75,19.50,3327,NaN,0,0.00


# 存成CSV檔


In [34]:
# 將檔案存檔成csv（可以用excel打開）
# 用dataframe存檔，避免中文亂碼，記得要將encoding='utf_8_sig'喔！
df.to_csv('daily_price.csv', encoding='utf_8_sig')

# 讀檔
# 我們指名 index 為 證券代號
df = pd.read_csv('daily_price.csv', index_col=['證券代號'])

print('index為證券代號')
print('     v')
df.head()

index為證券代號
     v


,成交股數,成交筆數,成交金額,開盤價,最高價,最低價,收盤價,漲跌價差,最後揭示買價,最後揭示買量,最後揭示賣價,最後揭示賣量,本益比
證券代號,,,,,,,,,,,,,
00400A,60780296,14112,877124134,14.64,14.74,14.08,14.55,0.31,14.54,190,14.55,394,0.0
00401A,17322548,1412,235236888,13.80,13.80,13.38,13.68,0.16,13.67,154,13.68,225,0.0
00403A,545746967,128975,5646466276,10.44,10.54,10.13,10.41,0.18,10.41,1031,10.42,1511,0.0
0050,187031371,355382,19494259576,105.00,105.35,102.80,104.15,1.95,104.15,1046,104.20,272,0.0
0051,120899,761,17204042,144.85,144.85,139.45,143.25,2.35,142.85,1,143.25,1,0.0


# 存到 sqlite3 中


In [ ]:
# 將 sql 通道打開
import sqlite3

conn = sqlite3.connect('test.sqlite3')

# 存檔 if_exists='replace' 是說假如sql中已經有 daily_price 這個 dataframe，則取代它
df.to_sql('daily_price', conn, if_exists='replace')

# 讀檔
df = pd.read_sql('select * from daily_price', conn, index_col=['證券代號'])
df.head()

,成交股數,成交筆數,成交金額,開盤價,最高價,最低價,收盤價,漲跌價差,最後揭示買價,最後揭示買量,最後揭示賣價,最後揭示賣量,本益比
證券代號,,,,,,,,,,,,,
00400A,60780296,14112,877124134,14.64,14.74,14.08,14.55,0.31,14.54,190,14.55,394,0.0
00401A,17322548,1412,235236888,13.80,13.80,13.38,13.68,0.16,13.67,154,13.68,225,0.0
00403A,545746967,128975,5646466276,10.44,10.54,10.13,10.41,0.18,10.41,1031,10.42,1511,0.0
0050,187031371,355382,19494259576,105.00,105.35,102.80,104.15,1.95,104.15,1046,104.20,272,0.0
0051,120899,761,17204042,144.85,144.85,139.45,143.25,2.35,142.85,1,143.25,1,0.0


# 總結一下剛剛教的function：

1. pd.to_numeric(series) <--- 將series轉型成數字。
2. df.apply(func) <--- 將 dataframe 中的每一條 series 都用 func 處理一番。
3. lambda x: y <--- 一個無名氏function，讀入 x 吐出 y。
4. df.set_index(col_name) <--- 將某個column直接變成index
5. df[x] <--- 選取 df 中的 x ，假如 x 是 a (list or series) of (string or boolean)，
   假如為 boolean，則長度得跟columns的數目一樣常喔！
